# DLGenAI Project Milestone-4
This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

Suggested Readings: 
- Hugging Face Multiple Choice Task
- LoRA  Documentation
- PyTorchSoftmax
Perform the following tasks on the data as part of the kaggle competition .
Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
- A = 0
- B = 1
- C = 2
- D = 3
- E = 4

What is the encoded numeric label for the row at index 150?

In [3]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

raw_answer = train.iloc[150]['answer']

encoded_label = label_mapping[raw_answer]

print(f"The raw answer is '{raw_answer}', which encodes to numeric label: {encoded_label}")

The raw answer is 'C', which encodes to numeric label: 2


Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
***str(prompt) + " \[SEP] " + str(option_B)***

What is the exact character length of this formatted input string?

In [4]:
row_0 = train.iloc[0]

prompt_0 = str(row_0['prompt'])
option_B_0 = str(row_0['B'])

formatted_input_B = prompt_0 + " [SEP] " + option_B_0

char_length = len(formatted_input_B)
print(f"Exact character length of the formatted input string: {char_length}")

Exact character length of the formatted input string: 407


*Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length*

*Since each question has five options, every row becomes five tokenized sequences.*

Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
\[1, 5, 128]

What is the value of the second dimension(give the code to calculate)?

In [5]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row_0 = train.iloc[0]
prompt_0 = str(row_0['prompt'])
options = ['A', 'B', 'C', 'D', 'E']

formatted_inputs = [prompt_0 + " [SEP] " + str(row_0[opt]) for opt in options]

tokenized_outputs = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids_tensor = tokenized_outputs['input_ids'].unsqueeze(0) 

second_dimension_value = input_ids_tensor.shape[1]

print(f"The structural shape of the tensor is: {list(input_ids_tensor.shape)}")
print(f"The value of the second dimension is: {second_dimension_value}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The structural shape of the tensor is: [1, 5, 128]
The value of the second dimension is: 5


(torch.Size([1, 5, 128]), torch.Size([5, 128]))

Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
**\[16, 5, 128]**

How many total token positions are in this tensor?

In [8]:
tensor_shape = [16, 5, 128]

total_positions_math = tensor_shape[0] * tensor_shape[1] * tensor_shape[2]

dummy_tensor = torch.empty(tensor_shape)
total_positions_tensor = dummy_tensor.numel()

print(f"Tensor Shape: {tensor_shape}")
print(f"Total token positions (calculated mathematically): {total_positions_math}")
print(f"Total token positions (via torch .numel()): {total_positions_tensor}")

Tensor Shape: [16, 5, 128]
Total token positions (calculated mathematically): 10240
Total token positions (via torch .numel()): 10240


## Multiple-Choice Model Outputs

*AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.*

Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
\[1, 5]

How many logits are produced for one question?

In [9]:
from transformers import AutoModelForMultipleChoice, AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_mc = AutoModelForMultipleChoice.from_pretrained(model_name)

row_0 = train.iloc[0]
prompt_0 = str(row_0['prompt'])
options = ['A', 'B', 'C', 'D', 'E']
formatted_inputs = [prompt_0 + " [SEP] " + str(row_0[opt]) for opt in options]

tokenized_outputs = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = tokenized_outputs['input_ids'].unsqueeze(0)
attention_mask = tokenized_outputs['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits  # Structural shape: [1, 5]
num_logits_produced = logits.shape[1]

print(f"Output logits tensor shape: {list(logits.shape)}")
print(f"Number of logits produced for one question: {num_logits_produced}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Output logits tensor shape: [1, 5]
Number of logits produced for one question: 5


Q6. Supervised Loss Tensor

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [10]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_mc = AutoModelForMultipleChoice.from_pretrained(model_name)

row_0 = train.iloc[0]
prompt_0 = str(row_0['prompt'])
options = ['A', 'B', 'C', 'D', 'E']
formatted_inputs = [prompt_0 + " [SEP] " + str(row_0[opt]) for opt in options]

tokenized_outputs = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = tokenized_outputs['input_ids'].unsqueeze(0)
attention_mask = tokenized_outputs['attention_mask'].unsqueeze(0)

label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
correct_label = torch.tensor([label_mapping[row_0['answer']]])

with torch.no_grad():
    outputs = model_mc(
        input_ids=input_ids, 
        attention_mask=attention_mask, 
        labels=correct_label
    )

loss_tensor = outputs.loss
num_dimensions = loss_tensor.ndim  # or len(loss_tensor.shape)

print(f"Loss Tensor Value: {loss_tensor.item()}")
print(f"Number of dimensions in the loss tensor: {num_dimensions}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loss Tensor Value: 1.5992454290390015
Number of dimensions in the loss tensor: 0


**LoRA for Efficient Fine-Tuning**

*LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.*

Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:

r = 8

lora_alpha = 16

target_modules = \["query", "value"]

lora_dropout = 0.1

bias = "none"

task_type = TaskType.SEQ_CLS


Count trainable parameters using:

sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [13]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [14]:
!pip install torchao>=0.16.0

In [14]:
from peft import LoraConfig, get_peft_model, TaskType

model_name = "bert-base-uncased"
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS  # Sequence classification task type
)

peft_model = get_peft_model(base_model, lora_config)

trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Number of trainable parameters: 295,681


**Preparing Data for Hugging Face Trainer**

Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand:

tokenized input_ids, attention_mask, and numeric labels

Q8. Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:

input_ids with shape \[5, 128]

attention_mask with shape \[5, 128]

labels as the encoded answer label

For the first dataset item, input_ids has shape:
\[5, 128]

How many tokenized choices are stored in input_ids?

In [15]:
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_100 = train.iloc[:100]

label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

formatted_dataset_dict = {
    "input_ids": [],
    "attention_mask": [],
    "labels": []
}

for idx, row in train_100.iterrows():
    prompt = str(row['prompt'])
    options = ['A', 'B', 'C', 'D', 'E']
    
    choices_inputs = [prompt + " [SEP] " + str(row[opt]) for opt in options]
    
    tokenized = tokenizer(
        choices_inputs,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    formatted_dataset_dict["input_ids"].append(tokenized["input_ids"].tolist())
    formatted_dataset_dict["attention_mask"].append(tokenized["attention_mask"].tolist())
    formatted_dataset_dict["labels"].append(label_mapping[row['answer']])

hf_dataset = Dataset.from_dict(formatted_dataset_dict)

first_item_input_ids = torch.tensor(hf_dataset[0]["input_ids"])
choices_count = first_item_input_ids.shape[0]

print(f"Shape of first item's input_ids: {list(first_item_input_ids.shape)}")
print(f"Number of tokenized choices stored in input_ids: {choices_count}")

Shape of first item's input_ids: [5, 128]
Number of tokenized choices stored in input_ids: 5


**Tiny Fine-Tuning and Inference**

In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Q9. Tiny LoRA Fine-Tuning

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:

max_length = 64

per_device_train_batch_size = 4

gradient_accumulation_steps = 1

max_steps = 4

What is the final global_step reported by the Trainer?

In [15]:
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoModelForMultipleChoice, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
peft_model = get_peft_model(base_model, lora_config)

train_32 = train.iloc[:32]
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

dataset_dict = {"input_ids": [], "attention_mask": [], "labels": []}

for idx, row in train_32.iterrows():
    prompt = str(row['prompt'])
    options = ['A', 'B', 'C', 'D', 'E']
    choices_inputs = [prompt + " [SEP] " + str(row[opt]) for opt in options]
    
    tokenized = tokenizer(
        choices_inputs,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )
    
    dataset_dict["input_ids"].append(tokenized["input_ids"].tolist())
    dataset_dict["attention_mask"].append(tokenized["attention_mask"].tolist())
    dataset_dict["labels"].append(label_mapping[row['answer']])

hf_dataset = Dataset.from_dict(dataset_dict)

training_args = TrainingArguments(
    output_dir="./tiny_lora_mc",
    per_device_train_batch_size=4,        # Batch size per device
    gradient_accumulation_steps=1,       # Update weights every step
    max_steps=4,                         # Hard cap optimization execution at 4 steps
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=hf_dataset,
)

train_result = trainer.train()

print(f"Final global_step reported by the Trainer: {train_result.global_step}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Step,Training Loss
1,1.514534
2,1.641454
3,1.615683
4,1.689843


Final global_step reported by the Trainer: 4


Q10. Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [16]:
import torch.nn.functional as F

row_0 = train.iloc[0]
prompt_0 = str(row_0['prompt'])
options = ['A', 'B', 'C', 'D', 'E']
choices_inputs = [prompt_0 + " [SEP] " + str(row_0[opt]) for opt in options]

tokenized_0 = tokenizer(
    choices_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

input_ids_0 = tokenized_0["input_ids"].unsqueeze(0).to(peft_model.device)
attention_mask_0 = tokenized_0["attention_mask"].unsqueeze(0).to(peft_model.device)

peft_model.eval()
with torch.no_grad():
    outputs = peft_model(input_ids=input_ids_0, attention_mask=attention_mask_0)

logits = outputs.logits

probabilities = F.softmax(logits, dim=-1).squeeze(0)  # Shape becomes [5]

prob_option_E = probabilities[4].item()

print(f"Logits output vector: {logits.cpu().numpy()}")
print(f"Softmax Probabilities: {probabilities.cpu().numpy()}")
print(f"Probability assigned to Option E: {prob_option_E:.4f}")

Logits output vector: [[0.19569872 0.16191983 0.18463567 0.16466874 0.14881441]]
Softmax Probabilities: [0.20494208 0.19813497 0.2026873  0.19868037 0.19555527]
Probability assigned to Option E: 0.1956
